# MIT 516 — Data Analytics for Cyber Security
## Assessment D: Anomaly Detection in Network Logs Using the UNSW-NB15 Dataset

**Unit:** MIT 516 Data Analytics for Cyber Security  
**Assessment:** D — Data Analytics Report and Presentation  
**Dataset:** UNSW-NB15 Network Packet Dataset  
**Track:** Track 4 — Anomaly Detection in Network Logs  

---
### Pipeline Overview
| Step | Task |
|------|------|
| 1 | Data Loading & Inspection |
| 2 | Data Preprocessing |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Anomaly Detection (Isolation Forest) |
| 5 | ML Classifier (Random Forest) |
| 6 | Model Evaluation |
| 7 | Security Visualisation |

---
## Step 0 — Imports & Configuration

In [3]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import os

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Preprocessing ───────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# ── Imbalanced data ─────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Anomaly Detection ───────────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from scipy.stats import zscore

# ── Machine Learning ────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# ── Evaluation ──────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
    precision_recall_curve,
    average_precision_score
)

# ── Global settings ─────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 55)
pd.set_option('display.float_format', '{:.4f}'.format)
RANDOM_STATE = 42

# Plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})
sns.set_palette('Set2')

print('All libraries loaded successfully.')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

All libraries loaded successfully.
pandas 3.0.3 | numpy 2.4.6


---
## Step 1 — Data Loading & Inspection
The UNSW-NB15 dataset is distributed as four CSV files with **no column headers**.  
Column names must be loaded separately from `UNSW-NB15_features.csv`.  
All four CSV parts are concatenated into a single DataFrame for unified analysis.

In [6]:
# ── 1.1  Load feature names ─────────────────────────────────────────────────
# The features file uses Windows-1252 encoding (cp1252)
features_meta = pd.read_csv('UNSW-NB15_features.csv', encoding='cp1252')
col_names = features_meta['Name'].str.strip().tolist()

print(f'Feature count: {len(col_names)}')
print('Columns:', col_names)

Feature count: 49
Columns: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']


In [7]:
# ── 1.2  Load and concatenate all four CSV parts ────────────────────────────
csv_files = [f'UNSW-NB15_{i}.csv' for i in range(1, 5)]
parts = []

for f in csv_files:
    if os.path.exists(f):
        part = pd.read_csv(f, header=None, names=col_names, low_memory=False)
        parts.append(part)
        print(f'  Loaded {f}: {part.shape[0]:,} rows')
    else:
        print(f'  WARNING: {f} not found — skipping')

df = pd.concat(parts, ignore_index=True)
print(f'\nCombined dataset shape: {df.shape}')
print(f'Total records: {len(df):,} | Total features: {df.shape[1]}')

  Loaded UNSW-NB15_1.csv: 700,001 rows
  Loaded UNSW-NB15_2.csv: 700,001 rows
  Loaded UNSW-NB15_3.csv: 700,001 rows
  Loaded UNSW-NB15_4.csv: 440,044 rows

Combined dataset shape: (2540047, 49)
Total records: 2,540,047 | Total features: 49


In [8]:
# ── 1.3  Initial inspection ─────────────────────────────────────────────────
print('=== HEAD ===')
display(df.head())

print('\n=== DATA TYPES & NON-NULL COUNTS ===')
df.info(verbose=True, show_counts=True)

=== HEAD ===


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,service,Sload,Dload,Spkts,Dpkts,swin,dwin,stcpb,dtcpb,smeansz,dmeansz,trans_depth,res_bdy_len,Sjit,Djit,Stime,Ltime,Sintpkt,Dintpkt,tcprtt,synack,ackdat,is_sm_ips_ports,ct_state_ttl,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.0011,132,164,31,29,0,0,dns,500473.9375,621800.9375,2,2,0,0,0,0,66,82,0,0,0.0000,0.0000,1421927414,1421927414,0.0170,0.0130,0.0000,0.0000,0.0000,0,0,0.0000,0.0000,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.0361,528,304,31,29,0,0,-,87676.0859,50480.1719,4,4,0,0,0,0,132,76,0,0,9.8910,10.6827,1421927414,1421927414,7.0050,7.5643,0.0000,0.0000,0.0000,0,0,0.0000,0.0000,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.0011,146,178,31,29,0,0,dns,521894.5313,636282.3750,2,2,0,0,0,0,73,89,0,0,0.0000,0.0000,1421927414,1421927414,0.0170,0.0130,0.0000,0.0000,0.0000,0,0,0.0000,0.0000,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.0012,132,164,31,29,0,0,dns,436724.5625,542597.1875,2,2,0,0,0,0,66,82,0,0,0.0000,0.0000,1421927414,1421927414,0.0430,0.0140,0.0000,0.0000,0.0000,0,0,0.0000,0.0000,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.0012,146,178,31,29,0,0,dns,499572.2500,609067.5625,2,2,0,0,0,0,73,89,0,0,0.0000,0.0000,1421927414,1421927414,0.0050,0.0030,0.0000,0.0000,0.0000,0,0,0.0000,0.0000,0,7,9,1,1,1,1,1,NaN,0



=== DATA TYPES & NON-NULL COUNTS ===
<class 'pandas.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 49 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   srcip             2540047 non-null  str    
 1   sport             2540047 non-null  object 
 2   dstip             2540047 non-null  str    
 3   dsport            2540047 non-null  str    
 4   proto             2540047 non-null  str    
 5   state             2540047 non-null  str    
 6   dur               2540047 non-null  float64
 7   sbytes            2540047 non-null  int64  
 8   dbytes            2540047 non-null  int64  
 9   sttl              2540047 non-null  int64  
 10  dttl              2540047 non-null  int64  
 11  sloss             2540047 non-null  int64  
 12  dloss             2540047 non-null  int64  
 13  service           2540047 non-null  str    
 14  Sload             2540047 non-null  float64
 15  Dload             2540

In [9]:
# ── 1.4  Summary statistics ─────────────────────────────────────────────────
print('=== DESCRIPTIVE STATISTICS (numeric features) ===')
display(df.describe())

=== DESCRIPTIVE STATISTICS (numeric features) ===


,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,Sload,Dload,Spkts,Dpkts,swin,dwin,stcpb,dtcpb,smeansz,dmeansz,trans_depth,res_bdy_len,Sjit,Djit,Stime,Ltime,Sintpkt,Dintpkt,tcprtt,synack,ackdat,is_sm_ips_ports,ct_state_ttl,ct_flw_http_mthd,is_ftp_login,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,Label
count,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,1191902.0000,1110168.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000,2540047.0000
mean,0.6588,4339.6001,36427.5937,62.7820,30.7668,5.1639,16.3294,36956447.5089,2450861.2163,33.2888,42.7266,150.0887,149.7459,1261701280.6217,1261766463.3286,124.2536,276.6719,0.0833,4242.1178,1589.0367,730.0755,1423260752.8712,1423260753.6362,193.3225,78.8248,0.0062,0.0033,0.0029,0.0017,0.2612,0.2346,0.0397,9.2070,8.9890,6.4391,6.9010,4.6421,3.5927,6.8459,0.1265
std,13.9249,56405.9947,161096.0355,74.6228,42.8509,22.5171,56.5947,118604331.2825,4224862.9416,76.2839,121.5020,125.4824,125.5438,1422026719.1774,1422139125.9758,151.9162,335.6166,0.3500,47500.5260,16910.3633,3438.5583,1134448.5358,1134448.3357,2779.1631,1433.1914,0.0462,0.0259,0.0239,0.0406,0.6831,0.7941,0.1997,10.8368,10.8225,8.1620,8.2051,8.4776,6.1744,11.2583,0.3324
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1421927377.0000,1421927414.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
25%,0.0010,200.0000,178.0000,31.0000,29.0000,0.0000,0.0000,135396.2656,11915.9375,2.0000,2.0000,0.0000,0.0000,0.0000,0.0000,60.0000,69.0000,0.0000,0.0000,0.0000,0.0000,1421952193.0000,1421952196.0000,0.0090,0.0060,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,2.0000,2.0000,2.0000,1.0000,1.0000,1.0000,0.0000
50%,0.0159,1470.0000,1820.0000,31.0000,29.0000,3.0000,4.0000,589303.7500,589317.8750,12.0000,12.0000,255.0000,255.0000,639725026.0000,638417164.0000,73.0000,89.0000,0.0000,0.0000,19.1249,2.6536,1424226977.0000,1424226978.0000,0.4683,0.4148,0.0006,0.0005,0.0001,0.0000,0.0000,0.0000,0.0000,5.0000,5.0000,3.0000,4.0000,1.0000,1.0000,2.0000,0.0000
75%,0.2146,3182.0000,14894.0000,31.0000,29.0000,7.0000,14.0000,2039922.7500,2925973.5000,44.0000,42.0000,255.0000,255.0000,2467159621.0000,2469410644.0000,132.0000,565.0000,0.0000,0.0000,413.7935,63.5086,1424245007.0000,1424245008.0000,7.3514,6.2021,0.0007,0.0006,0.0001,0.0000,0.0000,0.0000,0.0000,10.0000,10.0000,6.0000,7.0000,2.0000,1.0000,5.0000,0.0000
max,8786.6377,14355774.0000,14657531.0000,255.0000,254.0000,5319.0000,5507.0000,5988000256.0000,128761904.0000,10646.0000,11018.0000,255.0000,255.0000,4294958913.0000,4294953724.0000,1504.0000,1500.0000,172.0000,6558056.0000,1483830.9170,781221.1183,1424262068.0000,1424262069.0000,84371.4960,59485.3200,10.0375,4.5253,5.5122,1.0000,6.0000,36.0000,4.0000,67.0000,67.0000,67.0000,67.0000,67.0000,60.0000,67.0000,1.0000


In [10]:
# ── 1.5  Missing value audit ────────────────────────────────────────────────
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_report = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
null_report = null_report[null_report['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if null_report.empty:
    print('No missing values detected across all columns.')
else:
    print(f'Columns with missing values ({len(null_report)}):')
    display(null_report)

Columns with missing values (3):


,Missing Count,Missing %
attack_cat,2218764,87.3500
is_ftp_login,1429879,56.2900
ct_flw_http_mthd,1348145,53.0800


In [12]:
# ── 1.6  Class distribution ─────────────────────────────────────────────────
print('=== BINARY LABEL DISTRIBUTION ===')
label_counts = df['Label'].value_counts()
label_pct = df['Label'].value_counts(normalize=True) * 100
label_summary = pd.DataFrame({'Count': label_counts, 'Percentage': label_pct.round(2)})
label_summary.index = ['Normal (0)', 'Attack (1)'] if label_summary.index[0] == 0 else ['Attack (1)', 'Normal (0)']
display(label_summary)

print('\n=== ATTACK CATEGORY DISTRIBUTION ===')
attack_counts = df['attack_cat'].value_counts()
display(attack_counts.to_frame())

=== BINARY LABEL DISTRIBUTION ===


,Count,Percentage
Normal (0),2218764,87.3500
Attack (1),321283,12.6500



=== ATTACK CATEGORY DISTRIBUTION ===


,count
attack_cat,
Generic,215481
Exploits,44525
Fuzzers,19195
DoS,16353
Reconnaissance,12228
Fuzzers,5051
Analysis,2677
Backdoor,1795
Reconnaissance,1759


---
## Step 2 — Data Preprocessing
Preprocessing covers five key tasks:
1. **Drop irrelevant identifiers** (IP addresses, timestamps — not useful for ML)
2. **Handle missing values** (median imputation for numeric; mode for categorical)
3. **Encode categorical features** (`proto`, `service`, `state`) using Label Encoding
4. **Normalise numeric features** using StandardScaler
5. **Address class imbalance** using SMOTE on the training set

In [13]:
# ── 2.1  Drop non-informative identifier columns ────────────────────────────
drop_cols = ['srcip', 'dstip', 'sport', 'dsport', 'Stime', 'Ltime', 'attack_cat']
# Keep 'attack_cat' separate before dropping for multi-class reference later
attack_cat_series = df['attack_cat'].copy()

df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
print(f'Shape after dropping identifier columns: {df_clean.shape}')
print(f'Remaining columns: {df_clean.columns.tolist()}')

Shape after dropping identifier columns: (2540047, 42)
Remaining columns: ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'Label']


In [14]:
# ── 2.2  Handle missing values ──────────────────────────────────────────────
# Numeric columns: fill with median (robust to outliers)
# Categorical columns: fill with mode
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()

df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f'Missing values after imputation: {df_clean.isnull().sum().sum()}')
print(f'Categorical columns to encode: {cat_cols}')

Missing values after imputation: 0
Categorical columns to encode: ['proto', 'state', 'service', 'ct_ftp_cmd']


In [15]:
# ── 2.3  Label Encode categorical features ──────────────────────────────────
le = LabelEncoder()
label_encoders = {}

for col in cat_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f'  Encoded "{col}": {le.classes_[:5]}... ({len(le.classes_)} unique values)')

print(f'\nAll features are now numeric. Shape: {df_clean.shape}')

  Encoded "proto": ['3pc' 'a/n' 'aes-sp3-d' 'any' 'argus']... (135 unique values)
  Encoded "state": ['acc' 'clo' 'con' 'eco' 'ecr']... (16 unique values)
  Encoded "service": ['-' 'dhcp' 'dns' 'ftp' 'ftp-data']... (13 unique values)
  Encoded "ct_ftp_cmd": ['' '0' '1' '2' '3']... (9 unique values)

All features are now numeric. Shape: (2540047, 42)


In [ ]:
# ── 2.4  Feature / target split ─────────────────────────────────────────────
# The dataset uses 'Label' (capital L) — use the correct column name.
X = df_clean.drop(columns=['Label'])
y = df_clean['Label'].astype(int)

print(f'Feature matrix X: {X.shape}')
print(f'Target vector y: {y.shape}')
print(f'Class balance before SMOTE:\n{y.value_counts()}')

KeyError: "['label'] not found in axis"

In [ ]:
# ── 2.5  Train / test split (stratified) ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Training set: {X_train.shape[0]:,} records')
print(f'Test set:     {X_test.shape[0]:,} records')

# Standardise AFTER splitting to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('\nStandardScaler fitted on training data. Test data transformed (no leakage).')

In [ ]:
# ── 2.6  SMOTE — address class imbalance ────────────────────────────────────
print('Class distribution BEFORE SMOTE:')
print(pd.Series(y_train).value_counts())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print('\nClass distribution AFTER SMOTE:')
print(pd.Series(y_train_resampled).value_counts())
print(f'\nResampled training set size: {X_train_resampled.shape[0]:,} records')

In [ ]:
# ── 2.7  Visualise class imbalance before vs after ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Before SMOTE
before = pd.Series(y_train).value_counts()
axes[0].bar(['Normal (0)', 'Attack (1)'], before.values,
            color=['#2196F3', '#F44336'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Class Distribution — Before SMOTE', fontweight='bold')
axes[0].set_ylabel('Record Count')
for i, v in enumerate(before.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

# After SMOTE
after = pd.Series(y_train_resampled).value_counts()
axes[1].bar(['Normal (0)', 'Attack (1)'], after.values,
            color=['#2196F3', '#F44336'], edgecolor='white', linewidth=1.2)
axes[1].set_title('Class Distribution — After SMOTE', fontweight='bold')
axes[1].set_ylabel('Record Count')
for i, v in enumerate(after.values):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('Figure 1: SMOTE Class Rebalancing Effect', y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_smote_class_balance.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 3 — Exploratory Data Analysis (EDA)
EDA reveals the statistical structure of the dataset and highlights features most relevant to attack detection.  
Key outputs: class distribution chart, correlation heatmap, feature distributions by label, top feature importance preview.

In [ ]:
# ── 3.1  Overall class distribution pie chart ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart — binary label
counts = df['Label'].value_counts()
labels = [f'Normal\n{counts[0]:,}', f'Attack\n{counts[1]:,}']
axes[0].pie(counts.values, labels=labels, autopct='%1.1f%%',
             colors=['#2196F3', '#F44336'], startangle=140,
             wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Binary Label Distribution', fontweight='bold')

# Bar chart — attack categories
attack_df = df[df['Label'] == 1]['attack_cat'].value_counts()
colors_cat = sns.color_palette('Set2', len(attack_df))
axes[1].barh(attack_df.index, attack_df.values, color=colors_cat, edgecolor='white')
axes[1].set_xlabel('Record Count')
axes[1].set_title('Attack Category Breakdown', fontweight='bold')
for i, v in enumerate(attack_df.values):
    axes[1].text(v + 100, i, f'{v:,}', va='center', fontsize=8)

plt.suptitle('Figure 2: Dataset Class Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 3.2  Correlation heatmap (top 20 numeric features) ──────────────────────
# Select top 20 numeric features most correlated with label
corr_with_label = X.corrwith(y).abs().sort_values(ascending=False)
top20_features = corr_with_label.head(20).index.tolist()
top20_features.append('Label')  # include target

corr_matrix = df_clean[top20_features].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    annot_kws={'size': 7}, cbar_kws={'shrink': 0.8}
)
plt.title('Figure 3: Correlation Heatmap — Top 20 Features vs Label', fontsize=12, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('fig3_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 3.3  Feature distribution: Normal vs Attack (top 6 features) ────────────
top6 = corr_with_label.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, feat in enumerate(top6):
    ax = axes[idx]
    normal_data = df_clean[df_clean['Label'] == 0][feat].clip(upper=df_clean[feat].quantile(0.99))
    attack_data = df_clean[df_clean['Label'] == 1][feat].clip(upper=df_clean[feat].quantile(0.99))
    ax.hist(normal_data, bins=50, alpha=0.6, color='#2196F3', label='Normal', density=True)
    ax.hist(attack_data, bins=50, alpha=0.6, color='#F44336', label='Attack', density=True)
    ax.set_title(f'{feat}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('Figure 4: Feature Distributions — Normal vs Attack Traffic', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 3.4  Protocol and service analysis ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Protocol distribution by label
proto_label = df[['proto', 'Label']].copy()
proto_label['Label'] = proto_label['Label'].map({0: 'Normal', 1: 'Attack'})
top_protos = df['proto'].value_counts().head(8).index
proto_label_top = proto_label[proto_label['proto'].isin(top_protos)]
proto_counts = proto_label_top.groupby(['proto', 'Label']).size().unstack(fill_value=0)
proto_counts.plot(kind='bar', ax=axes[0], color=['#F44336', '#2196F3'], edgecolor='white')
axes[0].set_title('Protocol Distribution by Label', fontweight='bold')
axes[0].set_xlabel('Protocol')
axes[0].set_ylabel('Record Count')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Label')

# Service distribution (top 10)
service_counts = df['service'].value_counts().head(10)
axes[1].barh(service_counts.index, service_counts.values,
              color=sns.color_palette('Set2', 10), edgecolor='white')
axes[1].set_title('Top 10 Services in Dataset', fontweight='bold')
axes[1].set_xlabel('Record Count')

plt.suptitle('Figure 5: Protocol and Service Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_protocol_service.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 4 — Anomaly Detection
Two complementary unsupervised anomaly detection approaches are applied:

- **Z-Score Statistical Detection**: Flags records where any numeric feature deviates beyond 3 standard deviations from the mean — a classic statistical outlier approach.
- **Isolation Forest**: A tree-based ensemble model that isolates anomalies by randomly partitioning features. Anomalies require fewer splits to isolate, yielding lower anomaly scores. Well-suited to high-dimensional network traffic data.

Both methods operate **without labels**, making them applicable in real-world deployments where labelled data is unavailable.

In [ ]:
# ── 4.1  Z-Score anomaly detection ──────────────────────────────────────────
# Compute z-scores on a representative sample (100k records) for efficiency
sample_size = min(100_000, len(X))
X_sample = X.sample(n=sample_size, random_state=RANDOM_STATE)
y_sample = y.loc[X_sample.index]

z_scores = np.abs(zscore(X_sample.select_dtypes(include=np.number), nan_policy='omit'))
z_anomaly_mask = (z_scores > 3).any(axis=1)

z_anomalies = z_anomaly_mask.sum()
z_anomaly_rate = z_anomalies / sample_size * 100

# Overlap with actual attack labels
z_precision = (z_anomaly_mask & (y_sample.values == 1)).sum() / z_anomalies * 100 if z_anomalies > 0 else 0

print('=== Z-SCORE ANOMALY DETECTION RESULTS ===')
print(f'Sample size:         {sample_size:,}')
print(f'Anomalies detected:  {z_anomalies:,} ({z_anomaly_rate:.2f}% of sample)')
print(f'Overlap with actual attacks: {z_precision:.1f}%')

In [ ]:
# ── 4.2  Isolation Forest ────────────────────────────────────────────────────
# contamination = approximate proportion of anomalies in data
attack_rate = y_sample.mean()
print(f'Estimated contamination rate (attack proportion): {attack_rate:.4f}')

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=float(attack_rate),
    max_samples='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

X_sample_scaled = scaler.transform(X_sample)
iso_forest.fit(X_sample_scaled)

# Predictions: -1 = anomaly, 1 = normal → convert to 1/0
iso_pred = iso_forest.predict(X_sample_scaled)
iso_pred_binary = np.where(iso_pred == -1, 1, 0)   # 1 = anomaly flagged
iso_scores = iso_forest.decision_function(X_sample_scaled)  # lower = more anomalous

iso_detected = iso_pred_binary.sum()
iso_overlap = ((iso_pred_binary == 1) & (y_sample.values == 1)).sum()
iso_precision = iso_overlap / iso_detected * 100 if iso_detected > 0 else 0
iso_recall = iso_overlap / (y_sample.values == 1).sum() * 100

print('\n=== ISOLATION FOREST RESULTS ===')
print(f'Anomalies detected:  {iso_detected:,} ({iso_detected/sample_size*100:.2f}%)')
print(f'Overlap with true attacks: {iso_overlap:,}')
print(f'Unsupervised Precision (attack overlap): {iso_precision:.1f}%')
print(f'Unsupervised Recall  (attack coverage):  {iso_recall:.1f}%')

In [ ]:
# ── 4.3  Isolation Forest anomaly score visualisation ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Score distribution
normal_scores = iso_scores[y_sample.values == 0]
attack_scores = iso_scores[y_sample.values == 1]

axes[0].hist(normal_scores, bins=60, alpha=0.6, color='#2196F3', label='Normal', density=True)
axes[0].hist(attack_scores, bins=60, alpha=0.6, color='#F44336', label='Attack', density=True)
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.2, label='Decision boundary')
axes[0].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[0].set_ylabel('Density')
axes[0].set_title('Isolation Forest Score Distribution', fontweight='bold')
axes[0].legend()

# Confusion: Z-score vs Isolation Forest comparison
methods = ['Z-Score', 'Isolation Forest']
detected = [z_anomalies, iso_detected]
overlap = [
    int((z_anomaly_mask & (y_sample.values == 1)).sum()),
    int(iso_overlap)
]
x_pos = np.arange(len(methods))
width = 0.35
axes[1].bar(x_pos - width/2, detected, width, label='Total Flagged', color='#FF9800')
axes[1].bar(x_pos + width/2, overlap, width, label='True Attacks Caught', color='#4CAF50')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(methods)
axes[1].set_ylabel('Record Count')
axes[1].set_title('Anomaly Method Comparison', fontweight='bold')
axes[1].legend()

plt.suptitle('Figure 6: Anomaly Detection Results', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_anomaly_detection.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 5 — Machine Learning Classifier
A **Random Forest** classifier is trained on the SMOTE-resampled training data.  
Random Forest was selected because:
- It handles high-dimensional tabular data effectively without extensive feature engineering
- It is robust to feature scale differences and outliers
- It produces feature importance scores, enabling security analyst interpretation
- It has been consistently reported as top-performing on NSL-KDD and UNSW-NB15 in prior literature

A Decision Tree baseline is also trained for comparison.

In [ ]:
# ── 5.1  Train Random Forest ─────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print('Training Random Forest...')
rf_clf.fit(X_train_resampled, y_train_resampled)
print('Training complete.')

# ── 5.2  Train Decision Tree baseline ────────────────────────────────────────
dt_clf = DecisionTreeClassifier(
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
print('\nTraining Decision Tree baseline...')
dt_clf.fit(X_train_resampled, y_train_resampled)
print('Training complete.')

In [ ]:
# ── 5.3  Generate predictions on test set ────────────────────────────────────
rf_pred  = rf_clf.predict(X_test_scaled)
rf_proba = rf_clf.predict_proba(X_test_scaled)[:, 1]

dt_pred  = dt_clf.predict(X_test_scaled)
dt_proba = dt_clf.predict_proba(X_test_scaled)[:, 1]

print('Predictions generated on test set.')
print(f'RF — unique predicted classes: {np.unique(rf_pred)}')
print(f'DT — unique predicted classes: {np.unique(dt_pred)}')

---
## Step 6 — Model Evaluation
Models are evaluated using:
- **Confusion Matrix** — reveals true positives, false positives, false negatives, true negatives
- **Classification Report** — Precision, Recall, F1-Score per class
- **ROC-AUC Curve** — measures discriminative power across thresholds
- **Precision-Recall Curve** — critical for imbalanced class scenarios

In a cybersecurity context, **Recall (True Positive Rate)** for the attack class is the most operationally important metric — a missed attack (false negative) is typically more costly than a false alarm.

In [ ]:
# ── 6.1  Classification reports ──────────────────────────────────────────────
print('=' * 60)
print('RANDOM FOREST — Classification Report')
print('=' * 60)
print(classification_report(y_test, rf_pred, target_names=['Normal', 'Attack']))

print('=' * 60)
print('DECISION TREE — Classification Report')
print('=' * 60)
print(classification_report(y_test, dt_pred, target_names=['Normal', 'Attack']))

# AUC scores
rf_auc = roc_auc_score(y_test, rf_proba)
dt_auc = roc_auc_score(y_test, dt_proba)
print(f'Random Forest AUC-ROC: {rf_auc:.4f}')
print(f'Decision Tree AUC-ROC: {dt_auc:.4f}')

In [ ]:
# ── 6.2  Confusion matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in zip(
    axes,
    [rf_pred, dt_pred],
    ['Random Forest', 'Decision Tree (Baseline)']
):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Attack'])
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(f'{title}\nConfusion Matrix', fontweight='bold')

plt.suptitle('Figure 7: Confusion Matrices — Random Forest vs Decision Tree', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 6.3  ROC curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
RocCurveDisplay.from_predictions(
    y_test, rf_proba, name=f'Random Forest (AUC={rf_auc:.3f})',
    ax=axes[0], color='#2196F3'
)
RocCurveDisplay.from_predictions(
    y_test, dt_proba, name=f'Decision Tree (AUC={dt_auc:.3f})',
    ax=axes[0], color='#FF9800'
)
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[0].set_title('ROC Curve Comparison', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)

# Precision-Recall curves
rf_ap = average_precision_score(y_test, rf_proba)
dt_ap = average_precision_score(y_test, dt_proba)

rf_prec, rf_rec, _ = precision_recall_curve(y_test, rf_proba)
dt_prec, dt_rec, _ = precision_recall_curve(y_test, dt_proba)

axes[1].plot(rf_rec, rf_prec, color='#2196F3', linewidth=2, label=f'Random Forest (AP={rf_ap:.3f})')
axes[1].plot(dt_rec, dt_prec, color='#FF9800', linewidth=2, label=f'Decision Tree (AP={dt_ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Figure 8: ROC and Precision-Recall Curves', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_roc_pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 6.4  5-Fold Stratified Cross-Validation ───────────────────────────────────
# Use a subsample for cross-val speed on large datasets
cv_sample_size = min(50_000, len(X_train_resampled))
idx = np.random.choice(len(X_train_resampled), cv_sample_size, replace=False)
X_cv = X_train_resampled[idx]
y_cv = y_train_resampled[idx]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_cv_scores = cross_val_score(rf_clf, X_cv, y_cv, cv=skf, scoring='f1', n_jobs=-1)

print('=== 5-FOLD CROSS-VALIDATION (Random Forest — F1 Score) ===')
print(f'Fold scores: {np.round(rf_cv_scores, 4)}')
print(f'Mean F1:  {rf_cv_scores.mean():.4f}')
print(f'Std Dev:  {rf_cv_scores.std():.4f}')

In [ ]:
# ── 6.5  Performance comparison summary table ─────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

results = {
    'Model': ['Random Forest', 'Decision Tree', 'Isolation Forest (Unsupervised)'],
    'Accuracy': [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_sample, iso_pred_binary)
    ],
    'Precision': [
        precision_score(y_test, rf_pred),
        precision_score(y_test, dt_pred),
        precision_score(y_sample, iso_pred_binary)
    ],
    'Recall': [
        recall_score(y_test, rf_pred),
        recall_score(y_test, dt_pred),
        recall_score(y_sample, iso_pred_binary)
    ],
    'F1-Score': [
        f1_score(y_test, rf_pred),
        f1_score(y_test, dt_pred),
        f1_score(y_sample, iso_pred_binary)
    ],
    'AUC-ROC': [rf_auc, dt_auc, 'N/A (Unsupervised)']
}

results_df = pd.DataFrame(results)
print('=== MODEL PERFORMANCE SUMMARY ===')
display(results_df.set_index('Model').round(4))

---
## Step 7 — Security Visualisation Dashboard
A dashboard-quality figure summarising the full analytical pipeline for security operations audiences.  
Includes: attack category breakdown, top predictive features, model performance comparison, and anomaly score profile.

In [ ]:
# ── 7.1  Top 20 Feature Importances ──────────────────────────────────────────
feature_names = X.columns.tolist()
importances = rf_clf.feature_importances_
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
palette = sns.color_palette('viridis', len(feat_imp_df))
bars = plt.barh(feat_imp_df['Feature'][::-1], feat_imp_df['Importance'][::-1],
                color=palette[::-1], edgecolor='white')
plt.xlabel('Mean Decrease in Impurity (Gini Importance)')
plt.title('Figure 9: Top 20 Feature Importances — Random Forest', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('fig9_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 7.2  Security Operations Dashboard (composite figure) ────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#1a1a2e')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

TEXT_COLOR = 'white'
GRID_COLOR = '#444444'

# ─── Panel A: Attack category distribution ───
ax1 = fig.add_subplot(gs[0, 0])
attack_series = df[df['Label'] == 1]['attack_cat'].value_counts()
colors_a = plt.cm.plasma(np.linspace(0.2, 0.9, len(attack_series)))
ax1.barh(attack_series.index, attack_series.values, color=colors_a)
ax1.set_title('A. Attack Categories', color=TEXT_COLOR, fontweight='bold', fontsize=10)
ax1.tick_params(colors=TEXT_COLOR, labelsize=7)
ax1.set_facecolor('#16213e')
for spine in ax1.spines.values(): spine.set_edgecolor(GRID_COLOR)

# ─── Panel B: Top 10 feature importances ───
ax2 = fig.add_subplot(gs[0, 1])
top10_feat = feat_imp_df.head(10)
colors_b = plt.cm.YlOrRd(np.linspace(0.4, 0.9, 10))
ax2.barh(top10_feat['Feature'][::-1], top10_feat['Importance'][::-1], color=colors_b[::-1])
ax2.set_title('B. Top 10 Features (RF)', color=TEXT_COLOR, fontweight='bold', fontsize=10)
ax2.tick_params(colors=TEXT_COLOR, labelsize=7)
ax2.set_facecolor('#16213e')
for spine in ax2.spines.values(): spine.set_edgecolor(GRID_COLOR)

# ─── Panel C: Model accuracy comparison ───
ax3 = fig.add_subplot(gs[0, 2])
model_names = ['Random\nForest', 'Decision\nTree', 'Isolation\nForest']
acc_vals = [
    accuracy_score(y_test, rf_pred),
    accuracy_score(y_test, dt_pred),
    accuracy_score(y_sample, iso_pred_binary)
]
bar_colors = ['#00b4d8', '#90e0ef', '#caf0f8']
ax3.bar(model_names, acc_vals, color=bar_colors, edgecolor=GRID_COLOR)
ax3.set_ylim(0, 1.1)
ax3.set_title('C. Model Accuracy', color=TEXT_COLOR, fontweight='bold', fontsize=10)
ax3.tick_params(colors=TEXT_COLOR, labelsize=8)
ax3.set_facecolor('#16213e')
for i, v in enumerate(acc_vals): ax3.text(i, v + 0.01, f'{v:.3f}', ha='center', color=TEXT_COLOR, fontsize=8)
for spine in ax3.spines.values(): spine.set_edgecolor(GRID_COLOR)

# ─── Panel D: ROC curve ───
ax4 = fig.add_subplot(gs[1, 0:2])
RocCurveDisplay.from_predictions(y_test, rf_proba, name=f'Random Forest (AUC={rf_auc:.3f})', ax=ax4, color='#00b4d8')
RocCurveDisplay.from_predictions(y_test, dt_proba, name=f'Decision Tree (AUC={dt_auc:.3f})', ax=ax4, color='#f77f00')
ax4.plot([0, 1], [0, 1], 'w--', linewidth=0.8, alpha=0.5)
ax4.set_title('D. ROC Curves', color=TEXT_COLOR, fontweight='bold', fontsize=10)
ax4.tick_params(colors=TEXT_COLOR)
ax4.set_facecolor('#16213e')
ax4.legend(facecolor='#16213e', labelcolor=TEXT_COLOR, fontsize=8)
for spine in ax4.spines.values(): spine.set_edgecolor(GRID_COLOR)

# ─── Panel E: Isolation Forest score distribution ───
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(iso_scores[y_sample.values == 0], bins=50, alpha=0.7, color='#00b4d8', label='Normal', density=True)
ax5.hist(iso_scores[y_sample.values == 1], bins=50, alpha=0.7, color='#e63946', label='Attack', density=True)
ax5.axvline(0, color='white', linestyle='--', linewidth=1)
ax5.set_title('E. Isolation Forest Scores', color=TEXT_COLOR, fontweight='bold', fontsize=10)
ax5.tick_params(colors=TEXT_COLOR, labelsize=7)
ax5.set_facecolor('#16213e')
ax5.legend(facecolor='#16213e', labelcolor=TEXT_COLOR, fontsize=8)
for spine in ax5.spines.values(): spine.set_edgecolor(GRID_COLOR)

# ─── Panel F: F1 / Precision / Recall comparison ───
ax6 = fig.add_subplot(gs[2, :])
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
rf_metrics = [accuracy_score(y_test, rf_pred), precision_score(y_test, rf_pred),
              recall_score(y_test, rf_pred), f1_score(y_test, rf_pred)]
dt_metrics = [accuracy_score(y_test, dt_pred), precision_score(y_test, dt_pred),
              recall_score(y_test, dt_pred), f1_score(y_test, dt_pred)]
x_m = np.arange(len(metrics_names))
w = 0.3
ax6.bar(x_m - w/2, rf_metrics, w, label='Random Forest', color='#00b4d8')
ax6.bar(x_m + w/2, dt_metrics, w, label='Decision Tree', color='#f77f00')
ax6.set_xticks(x_m)
ax6.set_xticklabels(metrics_names, color=TEXT_COLOR, fontsize=10)
ax6.set_ylim(0, 1.15)
ax6.set_title('F. Full Metrics Comparison — Random Forest vs Decision Tree', color=TEXT_COLOR, fontweight='bold', fontsize=10)
ax6.tick_params(colors=TEXT_COLOR)
ax6.set_facecolor('#16213e')
for spine in ax6.spines.values(): spine.set_edgecolor(GRID_COLOR)
ax6.legend(facecolor='#16213e', labelcolor=TEXT_COLOR)
for i, (rv, dv) in enumerate(zip(rf_metrics, dt_metrics)):
    ax6.text(i - w/2, rv + 0.01, f'{rv:.3f}', ha='center', color=TEXT_COLOR, fontsize=8)
    ax6.text(i + w/2, dv + 0.01, f'{dv:.3f}', ha='center', color=TEXT_COLOR, fontsize=8)

fig.suptitle('MIT 516 — UNSW-NB15 Network Anomaly Detection Dashboard',
             fontsize=14, fontweight='bold', color=TEXT_COLOR, y=0.98)

plt.savefig('fig10_security_dashboard.png', bbox_inches='tight', dpi=150, facecolor='#1a1a2e')
plt.show()
print('Dashboard saved as fig10_security_dashboard.png')

---
## Pipeline Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Data Loading & Inspection | ✅ Complete |
| 2 | Data Preprocessing (Encoding, Scaling, SMOTE) | ✅ Complete |
| 3 | Exploratory Data Analysis | ✅ Complete |
| 4 | Anomaly Detection (Z-Score + Isolation Forest) | ✅ Complete |
| 5 | ML Classifier (Random Forest + DT Baseline) | ✅ Complete |
| 6 | Model Evaluation (CM, ROC, Precision-Recall, CV) | ✅ Complete |
| 7 | Security Visualisation Dashboard | ✅ Complete |

All figures saved to working directory for inclusion in the IEEE report.